This is going to be the plotting codes for all data pertaining to the North Polar Spur Survey

In [ ]:
#system config init

import numpy as np
import matplotlib.pyplot as plt
#import rafpy
#import ugradio
# import astropy.units as u
# import astropy.coordinates as coord
# import astropy.time as atime
# import astropy.table as atable
import glob
import os




In [ ]:
import sys
print(sys.executable)

In [ ]:
# -- Point this at wherever you copied the npz files --------------------------
DATA_DIR = "C:/Users/abook/OneDrive/Desktop/UG-ASTRO/lab4/data/april29/nps_data/."

# -- Load all sci files -------------------------------------------------------
files = np.sort(glob.glob(os.path.join(DATA_DIR, "nps_sci_*.npz")))
print("Found {} science files".format(len(files)))

# -- Sanity check keys and shape on first file --------------------------------
d = np.load(files[0])
print("Keys:", list(d.keys()))
print("spec shape:", d["spec_A_pol0"].shape)          # expect (1024,)
print("n_fft stored:", int(d["n_fft"]))               # expect 1024
print("nblocks_acc stored:", int(d["nblocks_acc"]))   # expect 1000
print("sample_rate: {:.3f} MHz".format(float(d["sample_rate_hz"]) / 1e6))
print("freq resolution: {:.1f} kHz".format(
    float(d["sample_rate_hz"]) / int(d["n_fft"]) / 1e3))
print("l={:.2f}  b={:.2f}  status={}".format(
    float(d["l_deg"]), float(d["b_deg"]), str(d["pointing_status"])))

In [ ]:
# -- Plot a single file - both polarisations, both switched freqs -------------

def plot_single_file(fpath):
    d = np.load(fpath)

    freq_A = d["freq_hz_A"] / 1e6   # MHz
    freq_B = d["freq_hz_B"] / 1e6

    l        = float(d["l_deg"])
    b        = float(d["b_deg"])
    alt      = float(d["alt_deg"])
    az       = float(d["az_deg"])
    status   = str(d["pointing_status"])
    jd_start = float(d["jd_window_start"])
    n_fft    = int(d["n_fft"])
    nblocks  = int(d["nblocks_acc"])
    sr_mhz   = float(d["sample_rate_hz"]) / 1e6
    freq_res_khz = (float(d["sample_rate_hz"]) / n_fft) / 1e3

    fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharey=False)
    fig.suptitle(
        "l={:.2f}  b={:.2f}  alt={:.1f}  az={:.1f}  status={}  JD={:.4f}\n"
        "N_FFT={}  nblocks={}  SR={:.2f} MHz  freq_res={:.1f} kHz".format(
            l, b, alt, az, status, jd_start,
            n_fft, nblocks, sr_mhz, freq_res_khz),
        fontsize=9)

    axes[0, 0].plot(freq_A, d["spec_A_pol0"], linewidth=0.7)
    axes[0, 0].set_title("Freq A ({:.3f} MHz)  Pol 0".format(float(d["freq_A_hz"]) / 1e6))

    axes[0, 1].plot(freq_A, d["spec_A_pol1"], linewidth=0.7, color="tab:orange")
    axes[0, 1].set_title("Freq A ({:.3f} MHz)  Pol 1".format(float(d["freq_A_hz"]) / 1e6))

    axes[1, 0].plot(freq_B, d["spec_B_pol0"], linewidth=0.7, color="tab:green")
    axes[1, 0].set_title("Freq B ({:.3f} MHz)  Pol 0".format(float(d["freq_B_hz"]) / 1e6))

    axes[1, 1].plot(freq_B, d["spec_B_pol1"], linewidth=0.7, color="tab:red")
    axes[1, 1].set_title("Freq B ({:.3f} MHz)  Pol 1".format(float(d["freq_B_hz"]) / 1e6))

    for ax in axes.flat:
        ax.set_xlabel("Frequency (MHz)")
        ax.set_ylabel("Power (arb.)")
        ax.axvline(float(d["center_freq_hz"]) / 1e6, color="gray",
                   linestyle="--", linewidth=0.8, label="HI rest")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

plot_single_file(files[4])
print("Name of file:" + files[4])
for i in files:
    print(i)

In [ ]:
# -- Summary grid - pol0 freq A only, one panel per file ----------------------
# With 1024 channels the plots are lighter to render so you can show more

ncols = 4
nrows = int(np.ceil(len(files) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4, nrows * 3))
axes = axes.flat

for ax, fpath in zip(axes, files):
    d = np.load(fpath)
    freq_A = d["freq_hz_A"] / 1e6
    ax.plot(freq_A, d["spec_A_pol0"], linewidth=0.6)
    ax.set_title("l={:.1f} b={:.1f}".format(
        float(d["l_deg"]), float(d["b_deg"])), fontsize=8)
    ax.axvline(float(d["center_freq_hz"]) / 1e6, color="gray",
               linestyle="--", linewidth=0.6)
    ax.set_xlabel("MHz", fontsize=7)
    ax.tick_params(labelsize=6)
    # Annotate status so bad pointings are immediately obvious
    if str(d["pointing_status"]) != "on_target":
        ax.set_facecolor("#fff0f0")
        ax.set_title("l={:.1f} b={:.1f} [{}]".format(
            float(d["l_deg"]), float(d["b_deg"]),
            str(d["pointing_status"])), fontsize=7, color="red")

for ax in list(axes)[len(files):]:
    ax.set_visible(False)

plt.suptitle("All pointings - Freq A Pol 0  (N_FFT=1024  nblocks=1000)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# -- Frequency-switched difference spectrum (A - B) ---------------------------
# With 1024 channels and 1000 blocks the noise floor should be well averaged.
# A real HI detection shows a positive bump near center_freq in the diff.

def plot_diff_spectrum(fpath):
    d = np.load(fpath)

    freq_A = d["freq_hz_A"] / 1e6
    freq_B = d["freq_hz_B"] / 1e6
    center = float(d["center_freq_hz"]) / 1e6

    # Interpolate B onto A's frequency grid before subtracting
    spec_B_pol0_interp = np.interp(freq_A, freq_B, d["spec_B_pol0"])
    spec_B_pol1_interp = np.interp(freq_A, freq_B, d["spec_B_pol1"])

    diff_pol0 = d["spec_A_pol0"] - spec_B_pol0_interp
    diff_pol1 = d["spec_A_pol1"] - spec_B_pol1_interp

    # Noise estimate from the edges of the band well away from the HI line
    edge_mask = (freq_A < center - 0.8) | (freq_A > center + 0.8)
    noise_pol0 = np.std(diff_pol0[edge_mask])
    noise_pol1 = np.std(diff_pol1[edge_mask])

    fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)
    fig.suptitle(
        "Freq-switched diff (A-B)  l={:.2f}  b={:.2f}  N_FFT={}  nblocks={}".format(
            float(d["l_deg"]), float(d["b_deg"]),
            int(d["n_fft"]), int(d["nblocks_acc"])))

    for ax, diff, noise, label, color in zip(
            axes,
            [diff_pol0, diff_pol1],
            [noise_pol0, noise_pol1],
            ["Pol 0", "Pol 1"],
            ["tab:blue", "tab:orange"]):
        ax.plot(freq_A, diff, linewidth=0.7, color=color)
        ax.axhline(0, color="gray", linewidth=0.6)
        ax.axhline(noise, color="gray", linestyle=":", linewidth=0.6, label="1-sigma noise")
        ax.axhline(-noise, color="gray", linestyle=":", linewidth=0.6)
        ax.axvline(center, color="red", linestyle="--", linewidth=0.8, label="HI rest")
        ax.set_title("{} - rms={:.2e}".format(label, noise))
        ax.set_xlabel("Frequency (MHz)")
        ax.set_ylabel("Delta Power (arb.)")
        ax.legend(fontsize=8)

    plt.tight_layout()
    plt.show()

plot_diff_spectrum(files[0])